1. Fine-tune (or zero-shot with a pipeline) a small BERT-family model for review sentiment; evaluate with Week
4's full metric set on a held-out split.

In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report
from transformers import pipeline

In [ ]:
df = pd.read_csv("reviews.csv")
print("First 5 Reviews:")
print(df.head())
print("\nClass Distribution:")
print(df["label"].value_counts())

First 5 Reviews:
                                              review     label
0                Delivery took longer than expected.  negative
1  Delivery arrived quickly. I frequently order f...  positive
2                         The product feels premium.  negative
3  Delivery took longer than expected. Ordered se...  positive
4  Delivery arrived quickly. Express delivery was...  positive

Class Distribution:
label
positive    26674
negative    13326
Name: count, dtype: int64


In [33]:
#train test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
print("\nTraining Samples :", len(train_df))
print("Testing Samples  :", len(test_df))


Training Samples : 32000
Testing Samples  : 8000


In [34]:
#load BERT
classifier = pipeline(task="sentiment-analysis",model="distilbert-base-uncased-finetuned-sst-2-english")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 450.19it/s]


In [35]:
#predict sentiment
predictions = []
for review in test_df["review"]:
    prediction = classifier(review)[0]["label"]
    prediction = prediction.lower()
    if prediction == "positive":
        predictions.append("positive")
    else:
        predictions.append("negative")

In [36]:
y_true = test_df["label"].str.lower()

In [37]:
#evaluation metrics
accuracy = accuracy_score(y_true, predictions)
precision = precision_score(y_true, predictions, pos_label="positive")
recall = recall_score(y_true, predictions, pos_label="positive")
f1 = f1_score(y_true, predictions, pos_label="positive")
cm = confusion_matrix(y_true, predictions)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print("\nConfusion Matrix")
print(cm)
print("\nClassification Report")
print(classification_report(y_true, predictions))

Accuracy : 0.7528
Precision: 0.8754
Recall   : 0.7336
F1 Score : 0.7983

Confusion Matrix
[[2108  557]
 [1421 3914]]

Classification Report
              precision    recall  f1-score   support

    negative       0.60      0.79      0.68      2665
    positive       0.88      0.73      0.80      5335

    accuracy                           0.75      8000
   macro avg       0.74      0.76      0.74      8000
weighted avg       0.78      0.75      0.76      8000



In [38]:
results = test_df.copy()
results["Predicted"] = predictions
print("\nSample Predictions")
print(results[["review", "label", "Predicted"]].head(10))


Sample Predictions
                                                  review     label Predicted
16733               The shopping experience was average.  positive  negative
11483  Shipping cost was affordable. Delivery arrived...  positive  positive
27296  Delivery arrived quickly. The product feels pr...  positive  positive
29962  Express delivery was very convenient. The prod...  positive  positive
34661                          Delivery arrived quickly.  positive  positive
37773  Ordered several items together. The product fe...  positive  positive
25776               The shopping experience was average.  negative  negative
26407                      Shipping cost was affordable.  positive  positive
8794   Delivery arrived quickly. Express delivery was...  positive  positive
14984                          Delivery arrived quickly.  positive  positive


2. Embed 1,000 reviews with sentence-transformers; run Week 6's K-Means on the vectors; name the complaint
themes you discover ('late delivery', 'sizing', 'quality praise').

In [39]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

In [40]:
df = df.head(1000)

In [41]:
#load sentence transformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1618.13it/s]


In [42]:
#converting reviews to embeddings
embeddings = model.encode(df["review"].tolist(),show_progress_bar=True)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches: 100%|██████████| 32/32 [00:05<00:00,  5.51it/s]


In [43]:
#k means clustering
k = 5
kmeans = KMeans(n_clusters=k,random_state=42,n_init=10)
df["cluster"] = kmeans.fit_predict(embeddings)

In [44]:
for cluster in sorted(df["cluster"].unique()):
    sample_reviews = df[df["cluster"] == cluster]["review"].head(10)
    for review in sample_reviews:
        print("-", review)
cluster_names = {
    0: "Late Delivery",
    1: "Affordable Shipping",
    2: "Premium Product",
    3: "Satisfied Customers",
    4: "Expensive Shipping"}
df["Theme"] = df["cluster"].map(cluster_names)
print("\nCluster Distribution")
print(df["Theme"].value_counts())

- Shipping cost was affordable. Delivery arrived quickly.
- Shipping cost was affordable. Delivery arrived quickly.
- Shipping cost was affordable. Delivery arrived quickly.
- Shipping cost was affordable. The product feels premium.
- Shipping cost was affordable. Delivery arrived quickly.
- Shipping charges were too expensive. Express delivery was very convenient. The product quality was disappointing. I frequently order from this store.
- Shipping charges were too expensive. Delivery took longer than expected. Ordered several items together. The product feels premium. I frequently order from this store.
- Shipping cost was affordable. I frequently order from this store.
- Shipping cost was affordable. Delivery arrived quickly. Ordered several items together.
- Shipping charges were too expensive. Delivery took longer than expected. Ordered several items together. The product feels premium. I frequently order from this store.
- The product feels premium.
- The product feels premium.
-

3. Build mini-RAG: embed ShopSmart's FAQ paragraphs, retrieve top-3 for a user question, assemble the
grounded prompt, and generate an answer. Test with a question NOT covered by the FAQ — does your
assistant say 'I don't know'?

In [45]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

In [ ]:
faq = pd.read_csv("faq.csv")
print(faq.head())

                       question  \
0   What is your return policy?   
1  How long does delivery take?   
2     How can I track my order?   
3   Do you offer free shipping?   
4        Can I cancel my order?   

                                              answer  
0  Products can be returned within 30 days of del...  
1  Standard delivery takes 3 to 5 business days. ...  
2  You can track your order using the tracking nu...  
3  Yes. Orders above Rs. 3000 qualify for free st...  
4  Orders can be cancelled before they are shippe...  


In [47]:
#load sentence transformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 717.02it/s]


In [48]:
#create embedding
documents = ("Question: " + faq["question"] +"\nAnswer: " + faq["answer"]).tolist()
document_embeddings = embedder.encode(documents,convert_to_numpy=True)

In [49]:
#load generation model
generator = pipeline("text-generation",model="google/flan-t5-base")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1650.41it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadMo

In [51]:
user_question = input("\nAsk ShopSmart a question: ")
query_embedding = embedder.encode(user_question,convert_to_numpy=True)
scores = cosine_similarity([query_embedding],document_embeddings)[0]
top_k = 3
top_indices = np.argsort(scores)[::-1][:top_k]
print("\nTop Retrieved FAQs\n")
context = ""
for i in top_indices:
    print(documents[i])
    print("Similarity:", round(scores[i], 3))
    context += documents[i] + "\n\n"
best_index = top_indices[0]
best_score = scores[best_index]
print("FINAL ANSWER")
if best_score < 0.35:
    print("answer is not available in the FAQ.")
else:
    print(faq.iloc[best_index]["answer"])


Top Retrieved FAQs

Question: What is your return policy?
Answer: Products can be returned within 30 days of delivery if they are unused and in their original packaging.
Similarity: 0.476
Question: Can I exchange a product?
Answer: Yes. Products can be exchanged within 30 days if they meet the exchange policy requirements.
Similarity: 0.299
Question: Can I cancel my order?
Answer: Orders can be cancelled before they are shipped. Once shipped, cancellation is not possible.
Similarity: 0.251
FINAL ANSWER
Products can be returned within 30 days of delivery if they are unused and in their original packaging.
